# YOLO Object Detection Training Notebook

This notebook demonstrates how to train an Ultralytics YOLOv8 model for object detection using the Vizon Backend framework.

**Contents:**
1. Install Dependencies
2. Prepare Dataset
3. Configure YOLO Model
4. Train the Model
5. Evaluate Performance
6. Save and Load Model

## 1. Install Ultralytics and Dependencies

First, we'll install all required libraries for YOLO training.

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    'ultralytics==8.0.0',
    'torch==2.0.0',
    'torchvision==0.15.0',
    'opencv-python==4.8.0.74',
    'numpy==1.24.3',
    'pyyaml==6.0',
]

print("Installing packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
print("✓ All packages installed successfully!")


In [ ]:
import os
os.chdir('/Users/enitze/vizon_backend')

# Verify installation and import modules
import torch
from ultralytics import YOLO
import sys
sys.path.insert(0, '/Users/enitze/vizon_backend')

from src.models import YOLOModel
from src.training import YOLOTrainer
from src.inference import YOLOPredictor
from utils import setup_seed, get_device

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("✓ All imports successful!")


## 2. Prepare Dataset

The dataset should be organized in YOLO format with separate directories for images and labels.


In [ ]:
import yaml
from pathlib import Path

# Create dataset directory structure
dataset_path = Path('datasets/custom_dataset')
for split in ['train', 'val', 'test']:
    (dataset_path / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dataset_path / 'labels' / split).mkdir(parents=True, exist_ok=True)

print("✓ Dataset directories created")

# Create example dataset configuration
dataset_config = {
    'path': str(dataset_path),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,  # Number of classes
    'names': ['object1', 'object2', 'object3']
}

# Save configuration
config_path = Path('configs/custom_dataset.yaml')
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(f"✓ Dataset config saved to {config_path}")
print("\nDataset structure created. You can now:")
print("1. Copy your images to datasets/custom_dataset/images/{train,val,test}/")
print("2. Copy your YOLO format labels to datasets/custom_dataset/labels/{train,val,test}/")


## 3. Configure YOLO Model

Now let's configure the YOLO model and hyperparameters for training.


In [ ]:
# Setup reproducibility
setup_seed(42)

# Get device
device = get_device(use_cuda=torch.cuda.is_available())

# Initialize YOLO model
print("Loading YOLOv8 model...")
model = YOLOModel(model_size='s', device=device.type)  # nano, small, medium, large, xlarge
print("✓ Model loaded successfully")
print(f"Model info: {model.get_model_info()}")

# Display training configuration
training_config = {
    'epochs': 50,
    'batch_size': 16,
    'imgsz': 640,
    'patience': 10,
    'lr0': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3
}

print("\nTraining Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")


## 4. Train the Model

Now we'll train the YOLO model on the prepared dataset. *(Note: This requires data in the dataset directory)*


In [ ]:
# Initialize trainer using the framework
trainer = YOLOTrainer(config_path='configs/training_config.yaml', log_dir='logs')

# Training code template
print("=" * 50)
print("Training Template")
print("=" * 50)
print("""
# To train your model, use:
results = trainer.train(
    data_yaml='configs/custom_dataset.yaml',  # Path to your dataset config
    epochs=50,
    batch_size=16,
    imgsz=640,
    patience=10,
    lr0=0.01,
    save_dir='runs/detect/custom_train'
)

print("Training completed!")
print(f"Best model saved to: runs/detect/custom_train/weights/best.pt")
""")

print("\n✓ Trainer initialized. Dataset required to run actual training.")
print("  Place your dataset in 'datasets/custom_dataset/' following YOLO format.")


## 5. Evaluate Model Performance

After training, evaluate the model on the validation set to compute metrics.


In [ ]:
from src.evaluation import YOLOEvaluator

# Create evaluator
model_for_eval = YOLOModel(model_size='s')
evaluator = YOLOEvaluator(model_for_eval)

print("=" * 50)
print("Evaluation Template")
print("=" * 50)
print("""
# To evaluate your trained model, use:
evaluator = YOLOEvaluator(model)

# Evaluate on test set
results = evaluator.evaluate(
    test_dir='datasets/custom_dataset/images/test',
    conf_threshold=0.5,
    iou_threshold=0.5
)

print(f"Total images evaluated: {results['total_images']}")
print(f"Total detections: {len(results['predictions'])}")

# Save evaluation results
evaluator.save_results('eval_results.json')
print("Results saved to eval_results.json")
""")

print("\n✓ Evaluator initialized. Trained model and test dataset required.")


## 6. Save and Load the Trained Model

Save your trained model and load it for inference.


In [ ]:
# Demonstrate model handling
print("=" * 50)
print("Model Management")
print("=" * 50)

# Load a pre-trained model
model = YOLOModel(model_size='s')
print("✓ Pre-trained model loaded")

# Export model to different formats
print("\nModel Export Options:")
print("  - format='onnx'  : ONNX format")
print("  - format='torchscript' : TorchScript format")
print("  - format='tflite' : TensorFlow Lite format")

print("\n# To export your trained model:")
print("""
export_path = model.export(format='onnx', half=False)
print(f"Model exported to: {export_path}")
""")

# Inference example
print("\n" + "=" * 50)
print("Inference Examples")
print("=" * 50)
print("""
# Load trained model
model = YOLOModel(model_size='s')
model.load_weights('runs/detect/custom_train/weights/best.pt')

# Create predictor
predictor = YOLOPredictor(model, conf_threshold=0.5)

# Predict on image
results = predictor.predict_image('test_image.jpg')
print(f"Detections: {len(results['detections'])}")

for det in results['detections']:
    print(f"  {det['class_name']}: {det['confidence']:.2f}")

# Predict on video
results = predictor.predict_video('video.mp4', output_path='output.mp4')

# Predict on webcam (30 seconds)
results = predictor.predict_webcam(duration=30, output_path='webcam.mp4')
""")

print("\n✓ Framework setup complete!")
print("Ready for training when you have your dataset prepared.")
